In [ ]:
import os
import pandas as pd
import numpy as np

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/cb_prediction/")
# REPO_DIR = "."
os.chdir(REPO_DIR)
data_dir = os.path.join(REPO_DIR, "Data")
temp_dir = os.path.join(REPO_DIR, "Temp")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)


In [18]:
def decode_embed_responses(responses):
    responses_decoded = []
    for _, response in responses.iterrows(): 
        response_out =  response.response['body']['data'][0]
        responses_decoded.append({
            "custom_id": response["custom_id"],
            "Embeddings": response_out['embedding'],
            "Tokens": int(response.response["body"]["usage"]["prompt_tokens"])
        })
    return(pd.json_normalize(responses_decoded))

In [39]:
description_embeddings = decode_embed_responses(pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_DescriptionClean.jsonl'), lines=True))

description_embeddings.rename(columns={
    'custom_id': 'BillID',
    'Embeddings': 'DescriptionCleanEmbed',
    'Tokens': 'DescriptionCleanTokens'
    }, inplace=True)


In [61]:
N = 10_000
samples = description_embeddings['DescriptionCleanEmbed'].sample(n=int(2*N), replace=True, random_state=123).reset_index(drop=True)

random_pairs = pd.DataFrame({
    "i": samples[:N].reset_index(drop=True),
    "j": samples[N:].reset_index(drop=True)
    }) 

rand_cosine = random_pairs.apply(lambda x: cosine_similarity(x["i"], x["j"]), axis=1).mean()
rand_euclidean = random_pairs.apply(lambda x: euclidean_distance(x["i"], x["j"]), axis=1).mean()

random_baseline = pd.DataFrame({
    "Metric": ["CosineSimilarity", "EuclideanDistance"],
    "RandomBaseline": [rand_cosine, rand_euclidean]
    }) 
print(random_baseline)

random_baseline_path = os.path.join(data_dir, "random_baseline.csv")
random_baseline.to_csv(random_baseline_path, index=False)
print(f"Saved {os.path.basename(random_baseline_path)}, n = {len(random_baseline)}, at {os.path.dirname(random_baseline_path)}")

              Metric  RandomBaseline
0   CosineSimilarity        0.379477
1  EuclideanDistance        1.110457
Saved random_baseline.csv, n = 2, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Data/Prediction


In [58]:
description_llm_embeddings = decode_embed_responses(pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_DescriptionLLMClean.jsonl'), lines=True))

description_llm_embeddings.rename(columns={
    'custom_id': 'ID',
    'Embeddings': 'DescriptionLLMCleanEmbed',
    'Tokens': 'DescriptionLLMCleanTokens'
    }, inplace=True)

bills_llm_completion = pd.read_csv(os.path.join(data_dir, f"bills_llm_completion.csv"))
bills_llm_completion = bills_llm_completion.merge(description_embeddings, on="BillID").merge(description_llm_embeddings, on="ID")

bills_llm_completion["TextSimilarity"] = bills_llm_completion["DescriptionClean"] == bills_llm_completion["DescriptionLLMClean"]
bills_llm_completion["EuclideanDistance"] = bills_llm_completion.apply(lambda x: euclidean_distance(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)
bills_llm_completion["CosineSimilarity"] = bills_llm_completion.apply(lambda x: cosine_similarity(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)

In [59]:
bills_llm_completion_similarity = bills_llm_completion[[
    'ID', 'BillID', 'PromptingStrategyID', 'PromptingStrategyName',
    'ResponseFormat', 'TrimText', 'AddIntrDate', 'Model', 'Temperature',
    'MaxTokens', 'Year', 'Major', 'MajorText', 'Party', 'Chamber', 'DW1', 'PassH', 'PassS', 'Postal', 'IntrDate',
    'DescriptionTrim', 'Description', 'DescriptionLLM', 'DescriptionClean', 'DescriptionLLMClean',
    'CosineSimilarity', 'EuclideanDistance', 'TextSimilarity']]
bills_llm_completion_similarity_path = os.path.join(data_dir, "bills_llm_completion_similarity.csv")
bills_llm_completion_similarity.to_csv(bills_llm_completion_similarity_path, index=False)
print(f"Saved {os.path.basename(bills_llm_completion_similarity_path)}, n = {len(bills_llm_completion_similarity)}, at {os.path.dirname(bills_llm_completion_similarity_path)}")

Saved bills_llm_completion_similarity.csv, n = 40000, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Data/Prediction
